<a href="https://colab.research.google.com/github/rmcpantoja/piper/blob/master/notebooks/piper_multilingual_training_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color="pink"> **[Piper](https://github.com/rhasspy/piper) training notebook.**
## ![Piper logo](https://contribute.rhasspy.org/img/logo.png)

---

- Notebook made by [rmcpantoja](http://github.com/rmcpantoja)
- Collaborator: [Xx_Nessu_xX](https://fakeyou.com/profile/Xx_Nessu_xX)

---

# Notes:

- <font color="orange">**Things in orange mean that they are important.**

# <font color="pink">🔧 ***First steps.*** 🔧

In [1]:
#@markdown ## <font color="pink"> **Check local Python environment.** 👁️
#@markdown ---
import sys
print('Python executable:', sys.executable)
print('Python version:', sys.version)

Python executable: /home/itelasoft/miniconda3/bin/python
Python version: 3.12.8 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:31:09) [GCC 11.2.0]


In [2]:
#@markdown ## <font color="pink"> **Check GPU type.** 👁️
#@markdown ---
#@markdown #### A higher capable GPU can lead to faster training speeds. By default, you will have a <font color="orange">**Tesla T4**</font>.
!nvidia-smi

Wed Apr 29 10:36:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.144.03             Driver Version: 550.144.03     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...    Off |   00000000:01:00.0  On |                  N/A |
|  0%   41C    P8             13W /  285W |     117MiB /  16376MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
#@markdown # <font color="pink"> **Offline mode notice.** 📂
#@markdown ---
print('Offline mode: no Google Drive mounting is required.')

Offline mode: no Google Drive mounting is required.


In [4]:
#@markdown # <font color="pink"> **Install software (offline-ready).** 📦

#@markdown #### This cell configures the local Piper repo and optionally installs Python dependencies if needed.
from pathlib import Path
import os

notebook_root = globals().get('notebook_root')
if notebook_root is None:
    cwd = Path.cwd()
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / 'piper' / 'src' / 'python').exists():
            notebook_root = candidate
            break
    if notebook_root is None:
        raise FileNotFoundError(
            'Unable to locate the notebook root. Place the Piper repository under <root>/piper.'
        )
piper_src = notebook_root / 'piper' / 'src' / 'python'
if not piper_src.exists():
    raise FileNotFoundError(
        f'Local Piper repo not found at {piper_src}. Place the Piper repository under {notebook_root}/piper.'
    )
print('Local Piper repository:', piper_src)
print('Notebook root:', notebook_root)
%cd "{piper_src}"

install_deps = False
if install_deps:
    !python -m pip install -q cython>=0.29.0 piper-phonemize==1.1.0 librosa>=0.9.2 numpy>=1.19.0 onnxruntime>=1.11.0 pytorch-lightning==1.7.0 torch==1.11.0 torchtext==0.12.0 torchvision==0.12.0 torchaudio==0.11.0 torchmetrics==0.11.4 pydub openai-whisper
    print('Dependencies installed.')
else:
    print('Skipping pip install. Ensure required packages are already installed in the local environment.')

# Build monotonic_align extension if missing
monotonic_dir = piper_src / 'piper_train' / 'vits' / 'monotonic_align'
so_files = list((monotonic_dir / 'monotonic_align').glob('core*.so')) if (monotonic_dir / 'monotonic_align').exists() else []
if not so_files:
    print('Building monotonic_align extension...')
    !bash "{piper_src / 'build_monotonic_align.sh'}"
    so_files = list((monotonic_dir / 'monotonic_align').glob('core*.so'))
    if not so_files:
        raise RuntimeError('Failed to build monotonic_align extension.')
    print('Built monotonic_align extension:', [str(p.name) for p in so_files])
else:
    print('monotonic_align extension already built:', [str(p.name) for p in so_files])

Local Piper repository: /home/itelasoft/Brianstorm/Projects/VoiceCloning/piper/src/python
Notebook root: /home/itelasoft/Brianstorm/Projects/VoiceCloning
/home/itelasoft/Brianstorm/Projects/VoiceCloning/piper/src/python
Skipping pip install. Ensure required packages are already installed in the local environment.
monotonic_align extension already built: ['core.cpython-312-x86_64-linux-gnu.so']


# <font color="pink"> 🤖 ***Training.*** 🤖

In [5]:
#@markdown # <font color="pink"> **1. Prepare local audio dataset.** 📥
#@markdown ---
#@markdown This cell converts a local .mp3 source into WAV segments and transcribes them with Whisper.
from pathlib import Path
import shutil
import csv

try:
    from pydub import AudioSegment
    from pydub.silence import split_on_silence
except ImportError:
    raise ImportError('pydub is required for audio conversion. Install it with pip install pydub')

try:
    import whisper
except ImportError:
    raise ImportError('whisper is required for transcription. Install it with pip install openai-whisper')

if shutil.which('ffmpeg') is None:
    raise RuntimeError('ffmpeg is required by pydub. Install ffmpeg and ensure it is on PATH.')

notebook_root = globals().get('notebook_root')
if notebook_root is None:
    cwd = Path.cwd()
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / 'piper' / 'src' / 'python').exists():
            notebook_root = candidate
            break
    if notebook_root is None:
        raise FileNotFoundError(
            'Unable to locate the notebook root. Place your source .mp3 file in the notebook folder or one of its parents.'
        )

search_dirs = [notebook_root] + list(notebook_root.parents)[:3]
source_audio_dir = None
mp3_files = []
for candidate in search_dirs:
    files = sorted(candidate.glob('*.mp3'))
    if files:
        source_audio_dir = candidate
        mp3_files = files
        break

if not mp3_files:
    raise FileNotFoundError(
        'No .mp3 files found in notebook root or parent directories. Place your source .mp3 file in the notebook folder or one of its parents.'
    )

print('Using MP3 from:', source_audio_dir)

dataset_dir = notebook_root / 'dataset'
wavs_dir = dataset_dir / 'wavs'
dataset_dir.mkdir(parents=True, exist_ok=True)
wavs_dir.mkdir(parents=True, exist_ok=True)

language = 'English (U.S.)'
languages = {
    'Català': 'ca',
    'Dansk': 'da',
    'Deutsch': 'de',
    'Ελληνικά': 'grc',
    'English (British)': 'en',
    'English (U.S.)': 'en-us',
    'Español': 'es',
    'Español (latinoamericano)': 'es-419',
    'Suomi': 'fi',
    'Français': 'fr',
    'Magyar': 'hu',
    'Icelandic': 'is',
    'Italiano': 'it',
    'ქართული': 'ka',
    'қазақша': 'kk',
    'Lëtzebuergesch': 'lb',
    'नेपाली': 'ne',
    'Nederlands': 'nl',
    'Norsk': 'nb',
    'Polski': 'pl',
    'Português (Brasil)': 'pt-br',
    'Română': 'ro',
    'Русский': 'ru',
    'Српски': 'sr',
    'Svenska': 'sv',
    'Kiswahili': 'sw',
    'Türkçe': 'tr',
    'украї́нська': 'uk',
    'Tiếng Việt': 'vi',
    '简体中文': 'zh',
}

final_language = languages[language]

sample_rate = 22050
split_ms = 15000
segments = []
for mp3_path in mp3_files:
    print('Converting', mp3_path.name)
    audio = AudioSegment.from_file(mp3_path)
    audio = audio.set_frame_rate(sample_rate).set_channels(1).set_sample_width(2)
    chunks = split_on_silence(
        audio,
        min_silence_len=700,
        silence_thresh=audio.dBFS - 14,
        keep_silence=250,
    )
    if len(chunks) > 1 and len(chunks) <= 20:
        for idx, chunk in enumerate(chunks, start=1):
            chunk_path = wavs_dir / f'{mp3_path.stem}_{idx:03d}.wav'
            chunk.export(chunk_path, format='wav')
            segments.append(chunk_path)
    else:
        for idx, start in enumerate(range(0, len(audio), split_ms), start=1):
            chunk = audio[start:start + split_ms]
            chunk_path = wavs_dir / f'{mp3_path.stem}_{idx:03d}.wav'
            chunk.export(chunk_path, format='wav')
            segments.append(chunk_path)

if not segments:
    raise RuntimeError('No WAV segments were created from the source mp3 files.')

print(f'Created {len(segments)} WAV segment(s) in {wavs_dir}')

model_name_whisper = 'small'
print('Loading Whisper model:', model_name_whisper)
whisper_model = whisper.load_model(model_name_whisper)

metadata_path = dataset_dir / 'metadata.csv'
with metadata_path.open('w', encoding='utf-8', newline='') as metadata_file:
    writer = csv.writer(metadata_file, delimiter='|', quoting=csv.QUOTE_MINIMAL)
    for wav_path in segments:
        print('Transcribing', wav_path.name)
        result = whisper_model.transcribe(str(wav_path), language='en' if final_language.startswith('en') else None)
        text = result['text'].strip()
        if not text:
            raise RuntimeError(f'Whisper returned empty transcription for {wav_path.name}')
        writer.writerow([f'wavs/{wav_path.name}', text])

print('Generated metadata.csv with', len(segments), 'entries.')

Using MP3 from: /home/itelasoft/Brianstorm/Projects/VoiceCloning
Converting KITT-3000.mp3
Created 17 WAV segment(s) in /home/itelasoft/Brianstorm/Projects/VoiceCloning/dataset/wavs
Loading Whisper model: small
Transcribing KITT-3000_001.wav
Transcribing KITT-3000_002.wav
Transcribing KITT-3000_003.wav
Transcribing KITT-3000_004.wav
Transcribing KITT-3000_005.wav
Transcribing KITT-3000_006.wav
Transcribing KITT-3000_007.wav
Transcribing KITT-3000_008.wav
Transcribing KITT-3000_009.wav
Transcribing KITT-3000_010.wav
Transcribing KITT-3000_011.wav
Transcribing KITT-3000_012.wav
Transcribing KITT-3000_013.wav
Transcribing KITT-3000_014.wav
Transcribing KITT-3000_015.wav
Transcribing KITT-3000_016.wav
Transcribing KITT-3000_017.wav
Generated metadata.csv with 17 entries.


In [6]:
#@markdown # <font color="pink"> **2. Transcript generation.** 📝
#@markdown ---
#@markdown The transcript is generated automatically from local WAV segments created from your .mp3 files in the previous step.
print('Transcript file will be generated automatically in dataset/metadata.csv.')

Transcript file will be generated automatically in dataset/metadata.csv.


In [7]:
#@markdown # <font color="pink"> **3. Preprocess dataset.** 🔄

import os
from pathlib import Path

#@markdown ### First of all, select the language of your dataset.
language = "English (U.S.)" #@param ["Català", "Dansk", "Deutsch", "Ελληνικά", "English (British)", "English (U.S.)", "Español", "Español (latinoamericano)", "Suomi", "Français", "Magyar", "Icelandic", "Italiano", "ქართული", "қазақша", "Lëtzebuergesch", "नेपाली", "Nederlands", "Norsk", "Polski", "Português (Brasil)", "Română", "Русский", "Српски", "Svenska", "Kiswahili", "Türkçe", "украї́нська", "Tiếng Việt", "简体中文"]
#@markdown ---
# language definition:
languages = {
    "Català": "ca",
    "Dansk": "da",
    "Deutsch": "de",
    "Ελληνικά": "grc",
    "English (British)": "en",
    "English (U.S.)": "en-us",
    "Español": "es",
    "Español (latinoamericano)": "es-419",
    "Suomi": "fi",
    "Français": "fr",
    "Magyar": "hu",
    "Icelandic": "is",
    "Italiano": "it",
    "ქართული": "ka",
    "қазақша": "kk",
    "Lëtzebuergesch": "lb",
    "नेपाली": "ne",
    "Nederlands": "nl",
    "Norsk": "nb",
    "Polski": "pl",
    "Português (Brasil)": "pt-br",
    "Română": "ro",
    "Русский": "ru",
    "Српски": "sr",
    "Svenska": "sv",
    "Kiswahili": "sw",
    "Türkçe": "tr",
    "украї́нська": "uk",
    "Tiếng Việt": "vi",
    "简体中文": "zh"
}


def _get_language(code):
    return languages[code]

final_language = _get_language(language)
#@markdown ### Choose a name for your model:
model_name = "Test" #@param {type:"string"}
#@markdown ---
notebook_root = globals().get('notebook_root')
if notebook_root is None:
    cwd = Path.cwd()
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / 'piper' / 'src' / 'python').exists():
            notebook_root = candidate
            break
    if notebook_root is None:
        raise FileNotFoundError(
            'Unable to locate the notebook root. Ensure the local Piper repo exists at <root>/piper/src/python.'
        )
print('Notebook root:', notebook_root)

output_path = notebook_root / 'piper_output' #@param {type:"string"}
output_dir = output_path / model_name
output_dir.mkdir(parents=True, exist_ok=True)
#@markdown ---
#@markdown ### Choose dataset format:
dataset_format = "ljspeech" #@param ["ljspeech", "mycroft"]
#@markdown ---
#@markdown ### Is this a single speaker dataset? Otherwise, uncheck:
single_speaker = True #@param {type:"boolean"}
if single_speaker:
  force_sp = " --single-speaker"
else:
  force_sp = ""
#@markdown ---
#@markdown ### Select the sample rate of the dataset:
sample_rate = "22050" #@param ["16000", "22050"]
#@markdown ---

piper_src = notebook_root / 'piper' / 'src' / 'python'
if not piper_src.exists():
    raise FileNotFoundError(f'Local Piper repo not found at {piper_src}')

%cd "{piper_src}"

#@markdown ### Do you want to train using this sample rate, but your audios don't have it?
#@markdown The resampler helps you do it quickly!
resample = False #@param {type:"boolean"}
if resample:
  !python resample.py --input_dir "{notebook_root / 'dataset' / 'wavs'}" --output_dir "{notebook_root / 'dataset' / 'wavs_resampled'}" --output_sr {sample_rate} --file_ext "wav"
  !rm -rf "{notebook_root / 'dataset' / 'wavs'}"
  !mv "{notebook_root / 'dataset' / 'wavs_resampled'}" "{notebook_root / 'dataset' / 'wavs'}"
#@markdown ---

max_workers = 1  # Use a single preprocess worker for small datasets to avoid zero batch size

monotonic_dir = piper_src / 'piper_train' / 'vits' / 'monotonic_align'
so_files = list((monotonic_dir / 'monotonic_align').glob('core*.so')) if (monotonic_dir / 'monotonic_align').exists() else []
if not so_files:
    print('monotonic_align extension missing, building now...')
    !bash "{piper_src / 'build_monotonic_align.sh'}"
    so_files = list((monotonic_dir / 'monotonic_align').glob('core*.so'))
    if not so_files:
        raise RuntimeError('Failed to build monotonic_align extension.')
    print('Built monotonic_align extension:', [str(p.name) for p in so_files])

!python -m piper_train.preprocess \
  --language {final_language} \
  --input-dir "{notebook_root / 'dataset'}" \
  --output-dir "{output_dir}" \
  --dataset-format {dataset_format} \
  --sample-rate {sample_rate} \
  --max-workers {max_workers} \
  {force_sp}

Notebook root: /home/itelasoft/Brianstorm/Projects/VoiceCloning
/home/itelasoft/Brianstorm/Projects/VoiceCloning/piper/src/python


INFO:preprocess:Single speaker dataset
INFO:preprocess:Wrote dataset config
INFO:preprocess:Processing 17 utterance(s) with 1 worker(s)


In [12]:
#@markdown # <font color="pink"> **4. Settings.** 🧰
import os
from pathlib import Path

notebook_root = globals().get('notebook_root')
if notebook_root is None:
    cwd = Path.cwd()
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / 'piper' / 'src' / 'python').exists():
            notebook_root = candidate
            break
    if notebook_root is None:
        raise FileNotFoundError(
            'Unable to locate the notebook root. Ensure the local Piper repo exists at <root>/piper/src/python.'
        )

import urllib.request

def download_checkpoint(url: str, output_path: Path) -> None:
    if output_path.exists():
        return
    print('Downloading pretrained checkpoint from:', url)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, output_path)
    if not output_path.exists():
        raise RuntimeError(f'Failed to download pretrained checkpoint to {output_path}')

action = "finetune" #@param ["Continue training", "convert single-speaker to multi-speaker model", "finetune", "train from scratch"]
pretrained_checkpoint_name = "joe.ckpt" #@param {type:"string"}
pretrained_checkpoint_url = "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/joe/medium/epoch%3D7889-step%3D1221224.ckpt"
pretrained_checkpoint_path = notebook_root / pretrained_checkpoint_name
if action in ("finetune", "convert single-speaker to multi-speaker model") and not pretrained_checkpoint_path.exists():
    download_checkpoint(pretrained_checkpoint_url, pretrained_checkpoint_path)
#@markdown ---
if action == "Continue training":
    ckpt = output_dir / 'lightning_logs' / 'version_0' / 'checkpoints' / 'last.ckpt'
    if ckpt.exists():
        ft_command = f'--resume_from_checkpoint "{ckpt}" '
        print(f'Continuing training from: {ckpt}')
    else:
        raise FileNotFoundError('No checkpoint found to continue training.')
elif action == "finetune":
    ckpt = pretrained_checkpoint_path
    if ckpt.exists():
        ft_command = f'--resume_from_checkpoint "{ckpt}" '
        print(f'Fine-tuning from pretrained checkpoint: {ckpt}')
    else:
        raise FileNotFoundError(f'{pretrained_checkpoint_name} not found in the notebook directory.')
elif action == "convert single-speaker to multi-speaker model":
    ckpt = pretrained_checkpoint_path
    if ckpt.exists():
        ft_command = f'--resume_from_single_speaker_checkpoint "{ckpt}" '
        print(f'Converting single-speaker checkpoint: {ckpt}')
    else:
        raise FileNotFoundError(f'{pretrained_checkpoint_name} not found in the notebook directory.')
else:
    ft_command = ""
    print('Training from scratch. No pretrained checkpoint will be used.')
#@markdown ### Choose batch size based on this dataset:
#@markdown * Use `1` if you are training on CPU with ~24 GB of RAM.
batch_size = 1 #@param {type:"integer"}
#@markdown ---
validation_split = 0.01
#@markdown ### Choose the quality for this model:
#@markdown * x-low - 16Khz audio, 5-7M params (recommended for CPU/24GB RAM)
#@markdown * medium - 22.05Khz audio, 15-20M params
#@markdown * high - 22.05Khz audio, 28-32M params
quality = "x-low" #@param ["high", "x-low", "medium"]
if action == "finetune" and "joe" in pretrained_checkpoint_url and "medium" in pretrained_checkpoint_url:
    print('Using quality=medium to match the Joe medium checkpoint architecture.')
    quality = "medium"
#@markdown ---
#@markdown ### For how many epochs to save training checkpoints?
#@markdown The larger your dataset should set this saving interval to a smaller value, as epochs can progress longer time.
checkpoint_epochs = 5 #@param {type:"integer"}
#@markdown ---
#@markdown ### Step interval to generate model samples:
log_every_n_steps = 1000 #@param {type:"integer"}
#@markdown ---
#@markdown ### Training epochs:
max_epochs = 10000 #@param {type:"integer"}
#@markdown ---

Fine-tuning from pretrained checkpoint: /home/itelasoft/Brianstorm/Projects/VoiceCloning/joe.ckpt
Using quality=medium to match the Joe medium checkpoint architecture.


In [16]:
#@markdown # <font color="pink"> **5. Train.** 🏋️‍♂️
#@markdown Run this cell to train your final model! If possible, some audio samples will be saved during training in the output folder.

monotonic_dir = piper_src / 'piper_train' / 'vits' / 'monotonic_align'
so_files = list((monotonic_dir / 'monotonic_align').glob('core*.so')) if (monotonic_dir / 'monotonic_align').exists() else []
if not so_files:
    print('monotonic_align extension missing, building now...')
    !bash "{piper_src / 'build_monotonic_align.sh'}"
    so_files = list((monotonic_dir / 'monotonic_align').glob('core*.so'))
    if not so_files:
        raise RuntimeError('Failed to build monotonic_align extension.')
    print('Built monotonic_align extension:', [str(p.name) for p in so_files])

get_ipython().system(f'''
python -m piper_train \
--dataset-dir "{output_dir}" \
--accelerator 'gpu' \
--devices 1 \
--batch-size {batch_size} \
--validation-split {validation_split} \
--num-test-examples 2 \
--quality {quality} \
--checkpoint-epochs {checkpoint_epochs} \
--save_last True \
--default_root_dir "{output_dir}" \
--log_every_n_steps 10 \
--max_epochs {max_epochs} \
{ft_command}\
--precision 32
''')
print('Training command launched with CPU-safe settings: quality=', quality, 'batch_size=', batch_size)

/home/itelasoft/miniconda3/lib/python3.12/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
DEBUG:piper_train:Namespace(dataset_dir='/home/itelasoft/Brianstorm/Projects/VoiceCloning/piper_output/Test', checkpoint_epochs=5, quality='medium', resume_from_single_speaker_checkpoint=None, batch_size=1, validation_split=0.01, num_test_examples=2, max_phoneme_ids=None, hidden_channels=192, inter_channels=192, filter_channels=768, n_layers=6, n_heads=2, lr_decay=0.999875, lr_reduce_enabled=False, lr_reduce_factor=0.5, lr_reduce_patience=10, show_plot=False, plot_save_path=None, learning_rate=0.0002, weight_decay=0.01, override_learning_rate=False, grad_clip=None, accelerator='gpu', devices=1, log_every

In [2]:
#@markdown # <font color="pink"> **6. Export ONNX model.** 🚀
import os
from pathlib import Path

notebook_root = globals().get('notebook_root')
if notebook_root is None:
    cwd = Path.cwd()
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / 'piper' / 'src' / 'python').exists():
            notebook_root = candidate
            break
    if notebook_root is None:
        raise FileNotFoundError(
            'Unable to locate the notebook root. Ensure the local Piper repo exists at <root>/piper/src/python.'
        )

output_path = notebook_root / 'piper_output'
model_name = globals().get('model_name', 'Test')
output_dir = output_path / model_name

checkpoint_paths = sorted(output_dir.glob('lightning_logs/version_*/checkpoints/*.ckpt'))
if not checkpoint_paths:
    raise FileNotFoundError(
        f'No checkpoint files found under {output_dir / "lightning_logs"}. '
        f'Please verify training saved checkpoints and rerun this cell.'
    )
checkpoint = checkpoint_paths[-1]
export_dir = output_dir / 'onnx'
export_dir.mkdir(parents=True, exist_ok=True)
onx_path = export_dir / f'{model_name}.onnx'
print('Exporting ONNX from', checkpoint)

piper_src = notebook_root / 'piper' / 'src' / 'python'
if not piper_src.exists():
    raise FileNotFoundError(f'Local Piper repo not found at {piper_src}')
%cd "{piper_src}"
!python -m piper_train.export_onnx "{checkpoint}" "{onx_path}"
print('ONNX export completed:', onx_path)


Exporting ONNX from /home/itelasoft/Brianstorm/Projects/VoiceCloning/piper_output/Test/lightning_logs/version_11/checkpoints/last.ckpt
/home/itelasoft/Brianstorm/Projects/VoiceCloning/piper/src/python


/home/itelasoft/miniconda3/lib/python3.12/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/home/itelasoft/miniconda3/lib/python3.12/site-packages/lightning_fabric/utilities/cloud_io.py:51: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary obj

# Have you finished training and want to test the model?

* If you want to run this model in any software that Piper integrates or the same Piper app, export your model using the [model exporter notebook](https://colab.research.google.com/github/rmcpantoja/piper/blob/master/notebooks/piper_model_exporter.ipynb)!
* Wait! I want to test this right now before exporting it to the supported format for Piper. Test your generated last.ckpt with [this notebook](https://colab.research.google.com/github/rmcpantoja/piper/blob/master/notebooks/piper_inference_(ckpt).ipynb)!